# Finding an optimal $\hat \eta$ using a grid search

In [13]:
from flow_helpers import *

In [ ]:
feathers = pathlib.Path(ds.__path__[0] + '/../../data/')

allpsrs = sorted(
    [ds.Pulsar.read_feather(psrfile) for psrfile in list(feathers.glob("*-[JB]*.feather"))],
    key=lambda psr: len(psr.toas), reverse = True
)

In [25]:
npsr = 1
psrs = allpsrs[2:2+npsr]

Tspan = ds.getspan(psrs)

In [26]:
rn_components = 30
gw_components = 14
powerlaw = ds.powerlaw

# here we fix the RN and WN, which should make the result exact, the WN is fixed through the psr.noisedict
rn_init_params = [{'log10_A': -11, 'gamma': 4}] * npsr #[{'log10_A': -11.0, 'gamma': 6.5}]*npsr # broad prior
tnequad = False
ecorr = False
fixed_wn = True

pslmodels = fouriermodel(psrs, rn_components, 
                                 rn_init_params= rn_init_params, ecorr = ecorr, tnequad = tnequad, 
                                 powerlaw = powerlaw, fixed_wn = fixed_wn)


In [27]:
psr_noisedicts = [psr.noisedict for psr in allpsrs]
L0s, ahat0_list, _ = compute_zero_quantities(pslmodels, psr_noisedicts)
L0s = jnp.stack(L0s)
ahat0_list = jnp.stack(ahat0_list)
Sigma_0_inv = jax.vmap(lambda L0: jnp.linalg.inv(L0 @ L0.T))(L0s)

In [28]:
b0s = jax.vmap(lambda L0, ahat0: jsp.linalg.cho_solve((L0, True), ahat0))(L0s, ahat0_list)

_, f, df = construct_freqs(psrs, num_frequencies=rn_components)

TtNT, _ = TtNT_mpsrs(Sigma_0_inv, params_list=[rn_init_params[0]] * npsr,
                                f=f, df=df, powerlaw=powerlaw)


In [29]:
b0s = jax.vmap(lambda L0, ahat0: jsp.linalg.cho_solve((L0, True), ahat0))(L0s, ahat0_list)

In [30]:
# Creating keys for the rn params + gw params
psrnames = [psr.name for psr in pslmodels]
rn_amp_keys, rn_gamma_keys = create_rn_keys(psrnames)

In [31]:
crn_gamma_key = "crn_gamma"
crn_log10A_key = "crn_log10_A"
crn_components = 14


In [32]:
for psr_idx in range(npsr):
    psrs_subset = [psrs[psr_idx]]

    rn_amp_key_subset = [rn_amp_keys[psr_idx]]    
    rn_gamma_key_subset = [rn_gamma_keys[psr_idx]]  

    commongp = ds.makecommongp_fourier(psrs_subset, ds.powerlaw, rn_components,
                                        T=Tspan, name='red_noise')
    commongp_crn = ds.makecommongp_fourier(psrs_subset, ds.powerlaw,
                                            components=crn_components, T=Tspan, name='crn',
                                            common=[crn_log10A_key, crn_gamma_key],)

    getN_common = commongp.Phi.getN
    getN_crn = commongp_crn.Phi.getN

    phi_crn_args = (crn_components, rn_amp_key_subset, rn_gamma_key_subset,
                    crn_log10A_key, crn_gamma_key, getN_common, getN_crn)

    phi_crn_partial = jax.jit(lambda rho: phi_crn(rho, *phi_crn_args))

    log_posterior = make_marginalized_log_posterior(
        TtNT[psr_idx], b0s[psr_idx], phi_crn_partial, 
        rn_amp_key_subset, rn_gamma_key_subset,
    crn_log10A_key, crn_gamma_key, n_crn_grid=10)

    eta_0 = eta_MAP(log_posterior,n_grid = 20, steps = 5, zoom = 0.3)

    print(f"eta MAP for {psrs[psr_idx].name}: {jnp.round(jnp.array(eta_0), 2)}")

eta MAP for J0636+5128: [  0.7 -13.3]


# Sensitivity to $\eta_0$. 

In [ ]:
psrs = [allpsrs[0], allpsrs[44]] # informative, uninformative psr

pslmodels_nonfixed = fouriermodel(psrs, rn_components, 
                                 rn_init_params= {}, ecorr = ecorr, tnequad = tnequad, 
                                 powerlaw = powerlaw, fixed_wn = True)